<a href="https://colab.research.google.com/github/poojithasiliveri2345-cmd/poojitha-Decision-tree/blob/main/fake_news_advanced.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#  Fake News Detection


## Setup Kaggle

In [ ]:
!pip install -q kaggle

In [ ]:
from google.colab import files
files.upload()

In [ ]:
import os
os.makedirs('/root/.kaggle', exist_ok=True)

In [ ]:
!cp kaggle.json /root/.kaggle/

In [ ]:
!chmod 600 /root/.kaggle/kaggle.json

##  Download Dataset

In [ ]:
!kaggle datasets download -d clmentbisaillon/fake-and-real-news-dataset

In [ ]:
!unzip -o fake-and-real-news-dataset.zip

##  Load Data

In [ ]:
import pandas as pd

In [ ]:
fake = pd.read_csv('Fake.csv')

In [ ]:
real = pd.read_csv('True.csv')

##  Label Data

In [ ]:
fake['label'] = 0

In [ ]:
real['label'] = 1

##  Merge Dataset

In [ ]:
df = pd.concat([fake, real])

In [ ]:
df = df.sample(frac=1).reset_index(drop=True)

##  Select Relevant Columns

In [ ]:
df = df[['text','label']]

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
df.isnull().sum()

##  EDA

In [ ]:
df['label'].value_counts()

In [ ]:
import seaborn as sns, matplotlib.pyplot as plt

In [ ]:
sns.countplot(x='label', data=df)
plt.show()

In [ ]:
df['length'] = df['text'].apply(len)

In [ ]:
df[['length']].describe()

In [ ]:
sns.histplot(df['length'], bins=50)
plt.show()

##  Text Cleaning

In [ ]:
import re, nltk
from nltk.corpus import stopwords

In [ ]:
nltk.download('stopwords')

In [ ]:
stop_words = set(stopwords.words('english'))

In [ ]:

def clean(text):
    text = re.sub('[^a-zA-Z]', ' ', text)
    text = text.lower().split()
    text = [w for w in text if w not in stop_words]
    return ' '.join(text)


In [ ]:
df['clean'] = df['text'].apply(clean)

##  Train-Test Split

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(df['clean'], df['label'], test_size=0.2, random_state=42)

##  TF-IDF Features

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

In [ ]:
tfidf = TfidfVectorizer(max_features=5000)

In [ ]:
X_train_tfidf = tfidf.fit_transform(X_train).toarray()

In [ ]:
X_test_tfidf = tfidf.transform(X_test).toarray()

##  Logistic Regression

In [ ]:
from sklearn.linear_model import LogisticRegression

In [ ]:
lr = LogisticRegression(max_iter=200)

In [ ]:
lr.fit(X_train_tfidf, y_train)

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
pred_lr = lr.predict(X_test_tfidf)

In [ ]:
accuracy_score(y_test, pred_lr)

##  Tokenization for LSTM

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [ ]:
tokenizer = Tokenizer(num_words=5000)

In [ ]:
tokenizer.fit_on_texts(X_train)

In [ ]:
X_train_seq = tokenizer.texts_to_sequences(X_train)

In [ ]:
X_test_seq = tokenizer.texts_to_sequences(X_test)

In [ ]:
X_train_pad = pad_sequences(X_train_seq, maxlen=200)

In [ ]:
X_test_pad = pad_sequences(X_test_seq, maxlen=200)

##  LSTM Model

In [ ]:
from tensorflow.keras.models import Sequential

In [ ]:
from tensorflow.keras.layers import Embedding, LSTM, Dense

In [ ]:

model = Sequential([
    Embedding(5000, 64, input_length=200),
    LSTM(64),
    Dense(1, activation='sigmoid')
])


In [ ]:
model.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])

##  Train LSTM

In [ ]:
history = model.fit(X_train_pad, y_train, epochs=3, validation_data=(X_test_pad, y_test))

## Evaluation

In [ ]:
model.evaluate(X_test_pad, y_test)

In [ ]:
plt.plot(history.history['accuracy'])

In [ ]:
plt.plot(history.history['val_accuracy'])

In [ ]:
plt.legend(['train','val'])

In [ ]:
plt.show()

##  Prediction Function

In [ ]:

def predict_news(text):
    cleaned = clean(text)
    seq = tokenizer.texts_to_sequences([cleaned])
    pad = pad_sequences(seq, maxlen=200)
    pred = model.predict(pad)
    return int(pred[0][0] > 0.5)

predict_news("Breaking: Major political event shocks the world")
